In [ ]:
import os
os.environ["POLARS_MAX_THREADS"] = "1"
os.environ["POLARS_STREAMING_CHUNK_SIZE"] = "30000"
import time
import random
import warnings

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from lxml import etree # type: ignore <- pylance milně hlásí chybu
from pathlib import Path
import time
import sys
import polars as pl
import polars.selectors as cs
import json
import pickle
import matplotlib.pyplot as plt

from utils import *
from schemas import *

pl.Config.set_tbl_cols(-1)
os.chdir(r'E:\CVUT_BAP')

In [ ]:
mereni_path = Path('kod/data/data_z_mericich_pristroju/parquet/mereni_all')
files = list(mereni_path.iterdir())
random.shuffle(files)

In [3]:
def short_display(df, pandas=True):
    list_cols = [name for name, dtype in df.schema.items() if isinstance(dtype, pl.List)]
    df_result = df.with_columns(pl.col(list_cols).list.len())
    if pandas:
        display(df_result.to_pandas())
    else:
        display(df_result)


def display_counts(df):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        display(df.null_count().to_pandas().T.astype(pd.Int64Dtype) * (-1) + len(df))

In [ ]:
df = None
for file in files:
    current = pl.read_parquet(file, schema=mereni_schema)
    if df is None:
        df = current
    else:
        df = pl.concat([df, current])
    if df.height > 2_000_000:
        break

if df is None:
    df = pl.read_parquet(mereni_path / 'Data z měřících přístrojů 04-01-2019.parquet', schema=mereni_schema)

In [ ]:
df_nafta = df.filter(pl.col('Benzin_PocetVyusteni').eq('0') & pl.col('Nafta_PocetVyusteni').ne('0') & pl.col('Plyn_PocetVyusteni').eq('0'))
display_counts(df_nafta)

,0
CisloProtokolu,1056040
DatumProhlidky,1056040
StaniceCislo,1056040
Zahajeni,1042075
Ukonceni,1042075
...,...
Plyn_OtackyZvysene_TPS_Vysledek,0
Plyn_Nadrz_Vyrobce,0
Plyn_Nadrz_Homologace,0
Plyn_Nadrz_Zivotnost,0


In [63]:
j1939_cols = [name for name in df_nafta.columns if 'J1939' in name]
df_nafta_personal = df_nafta.filter(pl.all_horizontal(pl.col(j1939_cols).is_null()))
display_counts(df_nafta_personal)

,0
CisloProtokolu,1043932
DatumProhlidky,1043932
StaniceCislo,1043932
Zahajeni,1030017
Ukonceni,1030017
...,...
Plyn_OtackyZvysene_TPS_Vysledek,0
Plyn_Nadrz_Vyrobce,0
Plyn_Nadrz_Homologace,0
Plyn_Nadrz_Zivotnost,0


In [86]:
obd_zazeh_cols = [name for name in df_nafta.columns if 'Obd_Readiness_Zazeh' in name]
df_nafta_personal_zazeh_readiness = df_nafta_personal.filter(pl.any_horizontal(pl.col(obd_zazeh_cols).is_not_null()))
short_display(df_nafta_personal_zazeh_readiness.select(pl.col('MericiPristroj_Typ').value_counts()).unnest('MericiPristroj_Typ'))
display_counts(df_nafta_personal_zazeh_readiness)
short_display(df_nafta_personal_zazeh_readiness)

,MericiPristroj_Typ,count
0,RTM430,7
1,OPA-100,15
2,AVL DiSmoke 480,16
3,AT605,6831


,0
CisloProtokolu,6869
DatumProhlidky,6869
StaniceCislo,6869
Zahajeni,6826
Ukonceni,6826
...,...
Plyn_OtackyZvysene_TPS_Vysledek,0
Plyn_Nadrz_Vyrobce,0
Plyn_Nadrz_Homologace,0
Plyn_Nadrz_Zivotnost,0


,CisloProtokolu,DatumProhlidky,StaniceCislo,Zahajeni,Ukonceni,OdpovednaOsoba,Prohlidka_CisloProtokolu,Prohlidka_DatumProhlidky,MericiPristroj_Vyrobce,MericiPristroj_Typ,...,Plyn_OtackyZvysene_NOX_Hodnota,Plyn_OtackyZvysene_NOX_Vysledek,Plyn_OtackyZvysene_O2_Hodnota,Plyn_OtackyZvysene_O2_Vysledek,Plyn_OtackyZvysene_TPS_Hodnota,Plyn_OtackyZvysene_TPS_Vysledek,Plyn_Nadrz_Vyrobce,Plyn_Nadrz_Homologace,Plyn_Nadrz_Zivotnost,Plyn_Nadrz_Kontrola
0,CZ-570711-22-03-0871,2022-03-25,570711,2022-03-25T17:52:37.5930000+01:00,2022-03-25T18:09:53.4670000+01:00,37267,CZ-570711-22-03-0871,2022-03-25T17:52:38,ACTIA CZ s.r.o.,AT605,...,None,None,None,None,None,None,None,None,None,None
1,CZ-570711-22-03-0869,2022-03-25,570711,2022-03-25T16:34:37.6430000+01:00,2022-03-25T16:50:47.0600000+01:00,50219,CZ-570711-22-03-0869,2022-03-25T16:34:38,ACTIA CZ s.r.o.,AT605,...,None,None,None,None,None,None,None,None,None,None
2,CZ-180601-22-03-0039,2022-03-25,180601,2022-03-25T15:10:12.3100000+01:00,2022-03-25T15:29:21.3000000+01:00,84637,CZ-180601-22-03-0039,2022-03-25T15:10:13,ACTIA CZ s.r.o.,AT605,...,None,None,None,None,None,None,None,None,None,None
3,CZ-570711-22-03-0862,2022-03-25,570711,2022-03-25T14:50:02.8730000+01:00,2022-03-25T15:07:46.3370000+01:00,37267,CZ-570711-22-03-0862,2022-03-25T14:50:03,ACTIA CZ s.r.o.,AT605,...,None,None,None,None,None,None,None,None,None,None
4,CZ-460305-22-03-0078,2022-03-25,460305,2022-03-25T14:18:56.5730000+01:00,2022-03-25T14:31:33.2570000+01:00,61025,CZ-460305-22-03-0078,2022-03-25T14:18:58,ACTIA CZ s.r.o.,AT605,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6864,CZ-440722-19-04-0392,2019-04-30,440722,2019-04-30T07:46:58.2900000+02:00,2019-04-30T08:00:49.3030000+02:00,35043,CZ-440722-19-04-0392,2019-04-30T07:46:50,ActiaCZ,AT605,...,None,None,None,None,None,None,None,None,None,None
6865,CZ-460252-19-04-0565,2019-04-30,460252,2019-04-30T07:27:05.6400000+02:00,2019-04-30T07:40:06.5130000+02:00,592,CZ-460252-19-04-0565,2019-04-30T07:27:04,ActiaCZ,AT605,...,None,None,None,None,None,None,None,None,None,None
6866,CZ-571007-19-04-0364,2019-04-30,571007,2019-04-30T07:47:38.9300000+02:00,2019-04-30T07:54:50.7330000+02:00,56871,CZ-571007-19-04-0364,2019-04-30T07:47:39,ActiaCZ,AT605,...,None,None,None,None,None,None,None,None,None,None
6867,CZ-471026-19-04-0285,2019-04-30,471026,2019-04-30T06:26:02.4030000+02:00,2019-04-30T06:34:31.4800000+02:00,4499,CZ-471026-19-04-0285,2019-04-30T06:25:59,ActiaCZ,AT605,...,None,None,None,None,None,None,None,None,None,None


In [87]:
df_nafta_personal_ok = df_nafta_personal.filter(pl.all_horizontal(pl.col(obd_zazeh_cols).is_null()))
display_counts(df_nafta_personal_ok)

,0
CisloProtokolu,1037063
DatumProhlidky,1037063
StaniceCislo,1037063
Zahajeni,1023191
Ukonceni,1023191
...,...
Plyn_OtackyZvysene_TPS_Vysledek,0
Plyn_Nadrz_Vyrobce,0
Plyn_Nadrz_Homologace,0
Plyn_Nadrz_Zivotnost,0


In [77]:
short_display(df_nafta_personal_ok.filter(pl.col('Vysledek_TesnostPlynovehoZarizeni').is_not_null()))

,CisloProtokolu,DatumProhlidky,StaniceCislo,Zahajeni,Ukonceni,OdpovednaOsoba,Prohlidka_CisloProtokolu,Prohlidka_DatumProhlidky,MericiPristroj_Vyrobce,MericiPristroj_Typ,...,Plyn_OtackyZvysene_NOX_Hodnota,Plyn_OtackyZvysene_NOX_Vysledek,Plyn_OtackyZvysene_O2_Hodnota,Plyn_OtackyZvysene_O2_Vysledek,Plyn_OtackyZvysene_TPS_Hodnota,Plyn_OtackyZvysene_TPS_Vysledek,Plyn_Nadrz_Vyrobce,Plyn_Nadrz_Homologace,Plyn_Nadrz_Zivotnost,Plyn_Nadrz_Kontrola
0,CZ-471413-25-08-0479,2025-08-15,471413,2025-08-15T09:49:08.2930000+02:00,2025-08-15T10:02:14.6370000+02:00,84627,CZ-471413-25-08-0479,2025-08-15T09:58:27.7555922+02:00,Bosch,BEA 950,...,None,None,None,None,None,None,None,None,None,None
1,CZ-460313-23-02-0292,2023-02-16,460313,2023-02-16T17:11:48.9400000+01:00,2023-02-16T17:34:14.5600000+01:00,35135,CZ-460313-23-02-0292,2023-02-16T17:32:13.1732749+01:00,Bosch,BEA 950,...,None,None,None,None,None,None,None,None,None,None


In [84]:
short_display(df_nafta_personal_ok.select(pl.col('Vozidlo_Palivo').value_counts()).unnest('Vozidlo_Palivo'))
short_display(df.select(pl.col('Vozidlo_Palivo').value_counts()).unnest('Vozidlo_Palivo'))

,Vozidlo_Palivo,count
0,BNB,12
1,NM,870688
2,N,166361
3,N+P,2


,Vozidlo_Palivo,count
0,N+P,4
1,CNG,5747
2,LNG,45
3,BA + LPG,28604
4,BA+LPG,3070
5,BA 96,1
6,BA 95/LP,7
7,BA 95/E85,6
8,BA 98,106
9,None,6171
